# Radiosonde/Dropsonde Zarr Pipeline
Demonstration of Zarr pipeline using M2HATS data, written for users to understand the process. This will be put into .py files and done behind the scenes. 

---

# Import required packages

In [13]:
# For analysis code
from pathlib import Path
import glob
import xarray as xr
import zarr

# For Dask + cluster
from dask_jobqueue import PBSCluster
from distributed import Client
from dask import delayed

---

# Designate a scratch directory
Define the designated scratch directory to hold Zarr stores created from field campaign data.

In [3]:
lustre_scratch  = "/lustre/desc1/scratch/myasears"

---

# Spin up a cluster
Create a cluster and scale it to 5 workers to assist with the processing in this notebook.

In [4]:
cluster = PBSCluster(
        job_name = 'dask-eol-25',
        cores = 1,
        memory = '4GiB',
        processes = 1,
        local_directory = lustre_scratch + '/dask/spill',
        log_directory = lustre_scratch + '/dask/logs/',
        resource_spec = 'select=1:ncpus=1:mem=4GB',
        queue = 'casper',
        walltime = '3:00:00',
        interface = 'ext')

client = Client(cluster)

In [ ]:
n_workers = 5
cluster.scale(n_workers)
client.wait_for_workers(n_workers = n_workers)

---

# Process data

### Create a Zarr store
I created a Zarr store in my scratch directory to contain the individual radiosonde datasets. 

In [60]:
zarr_path = "/lustre/desc1/scratch/myasears/sounding_data/zarr/m2hats"
root = zarr.open_group(zarr_path, mode = "a")

### Load data
This M2HATS radiosonde dataset was downloaded from the EOL Field Data Archive as 122 netcdf files, then stored in my scratch directory. I only downloaded ascending files for this case study. 

In [61]:
radiosonde_path = Path("/lustre/desc1/scratch/myasears/sounding_data/m2hats")
radiosonde_files = sorted([p for p in radiosonde_path.iterdir() if p.suffix == ".nc"])

### Populate the Zarr store
I iterate through every netcdf file, open it with xarray, drop incompatible variables, and convert them to Zarr.

In [62]:
for sonde in radiosonde_files:
    sounding_id = sonde.stem
    ds = xr.open_dataset(sonde)
    ds = ds.drop_vars("trajectory") # Drop "trajectory" - data type does not work with Zarr V3
    ds.to_zarr(zarr_path, group=sounding_id, consolidated=False)

---

# Open Zarr
See whether the Zarr store has been populated -- it has.

In [63]:
m2hats_zarr = "/lustre/desc1/scratch/myasears/sounding_data/zarr/m2hats"

In [68]:
root = zarr.open_group(m2hats_zarr, mode="r")
sounding_ids = list(root.group_keys())
sounding_ids

['NCAR_M2HATS_ISS1_RS41_v1_20230724_220030_asc',
 'NCAR_M2HATS_ISS1_RS41_v1_20230723_215950_asc',
 'NCAR_M2HATS_ISS1_RS41_v1_20230721_213732_asc',
 'NCAR_M2HATS_ISS1_RS41_v1_20230723_172736_asc',
 'NCAR_M2HATS_ISS1_RS41_v1_20230719_161344_asc']